<a href="https://colab.research.google.com/github/JoaoVitorCoelhoG/Aprendizado-Profundo/blob/main/cm204_lab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Instituto Tecnológico de Aeronáutica – ITA**

**Aprendizado Profundo – CM-204**

**Professor:**

Marcos Ricardo Omena de Albuquerque Maximo

# Laboratório 5 - Generative Adversarial Network (GAN)

**Instruções:**

Antes de submeter seu laboratório, certifique-se de que tudo está executando corretamente (em sequência): primeiro, **reinicie o kernel** (`Runtime->Restart Runtime` no Colab ou `Kernel->Restart` no Jupyter). Em seguida, execute todas as células (`Runtime->Run All` no Colab ou `Cell->Run All` no Jupyter) e verifique se todas as células executam sem erros, especialmente aquelas de correção automática, isto é, as que contêm `assert`s.

**Não delete as células de resposta**, isto é, aquelas que contêm `WRITE YOUR CODE HERE` ou `WRITE YOUR ANSWER HERE`, pois elas contêm metadados com os IDs das células para o sistema de correção. Pela mesma razão, **não delete as células de teste**, isto é, aquelas com `assert`s. Além disso, mantenha suas soluções dentro dos espaços reservados.

Os notebooks foram implementados para serem compatíveis com o Google Colab, instalando automaticamente as dependências e baixando os datasets. Os comandos que começam com `!` (ponto de exclamação) são comandos bash e podem ser executados em um terminal Linux.

---

## 1. Introdução

Neste laboratório, você implementará a técnica DCGAN (Deep Convolutional Generative Adversarial Network) para gerar imagens falsas de gatos.

Este laboratório foi baseado em diversos tutoriais disponíveis na Internet. Portanto, incentivo que você tente implementar as funções sem procurar tutoriais de DCGAN. É claro que você pode consultar as documentações das bibliotecas e outros sites. Além disso, não copie código de tutoriais. Lembre-se de que a intenção aqui é que você aprenda.

# 2. Atividades

## Instalações e Configurações

A célula a seguir instala dependências e configura o notebook para a implementação do laboratório.

In [ ]:
%pip install numpy matplotlib torch torchvision tqdm opencv-python gdown ipywidgets jupyter

## Imports

A célula a seguir importa as bibliotecas necessárias.

In [ ]:
import  os
import random
import numpy as np
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as tt
import torch
import torch.nn as nn
import cv2
from tqdm import tqdm
import torch.nn.functional as F
from torchvision.utils import save_image
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import random
%matplotlib inline

In [ ]:
# This cell defines a function for resetting the random seeds

def reset_seeds(seed=42):
  # 42 is the answer to the Ultimate Question of Life, the Universe, and Everything
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed) # In PyTorch, the CPU and GPU have separate random seeds, so we need to set both for reproducibility

In [ ]:
reset_seeds()

## Baixando o Conjunto de Dados

A célula a seguir baixa o conjunto de dados (*dataset*) com várias fotos de gatinhos fofinhos 😺.

In [ ]:
import zipfile

archive_path = 'archive.zip'
archive_url = 'https://drive.google.com/uc?id=1WrW8nXvYFafz5qNY9Mi8tU32Y-Z2M0Zd'

if not os.path.exists(archive_path):
    try:
        import gdown
        gdown.download(archive_url, archive_path, quiet=False)
    except Exception:
        import subprocess
        subprocess.check_call(['gdown', archive_url])

if not os.path.exists('cats'):
    with zipfile.ZipFile(archive_path, 'r') as archive:
        archive.extractall('.')

In [ ]:
DATA_DIR = '.'

## Visualizando o Conjunto de Dados

A célula a seguir mostra algumas imagens do conjunto de dados. Cuidado com a fofura! 😻

In [ ]:
grid_size = 8
num_imgs = grid_size * grid_size
fig = plt.figure(figsize=(grid_size, grid_size))
for num, fn in enumerate(os.listdir(DATA_DIR + '/cats')[:num_imgs]):
    path = DATA_DIR + '/cats/' + fn
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(grid_size, grid_size, num + 1)
    plt.axis('off')
    plt.imshow(img)

## Criando o Data Loader

A célula a seguir cria um data loader que redimensiona e normaliza as imagens.

In [ ]:
image_size = 64
batch_size = 128
normalize_mean = [0.5, 0.5, 0.5]
normalize_std = [0.5, 0.5, 0.5]

train_ds = ImageFolder(DATA_DIR + '/cats', transform=tt.Compose([
    tt.Resize(image_size),
    tt.CenterCrop(image_size),
    tt.ToTensor(),
    tt.Normalize(mean=normalize_mean, std=normalize_std)
]))

train_dl = DataLoader(train_ds, batch_size, shuffle=True, num_workers=2)

In [ ]:
# Defining some variables

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
latent_size = 100 # the dimension of the latent space used in the generator
size_multiplier = 64 # multiplier used to define the number of filters at each layer
num_color_channels = 3 # number of color channels of the input image
learning_rate = 0.0002 # learning rate used in the optimization algorithm
beta1 = 0.5 # hyperparameter beta1 of the Adam optimization algorithm
beta2 = 0.999 # hyperparameter beta2 of the Adam optimization algorithm

print(f'Using device: {device}')

## Implementando o Gerador

Na célula a seguir, implemente a rede geradora.

Instruções:
- A arquitetura da rede é baseada na arquitetura DCGAN, mas não é exatamente a mesma.
- Cinco camadas de convolução transposta são utilizadas para converter o espaço latente em uma imagem 3x64x64.
- Cada camada de convolução transposta utiliza tamanho de kernel igual a 4.
- O *stride* da primeira camada de convolução transposta é 1, enquanto as demais utilizam *stride* igual a 2.
- Nenhum *padding* é utilizado na primeira camada de convolução transposta, mas as subsequentes utilizam *padding* igual a 1.
- Nenhum *bias* é utilizado nas camadas de convolução transposta.
- As camadas intermediárias utilizam a função de ativação `ReLU`, enquanto a última utiliza `Tanh`.
- *Batch Normalization* é utilizada após cada camada de convolução transposta, exceto a última.
- Utilize `size_multiplier` para ajudá-lo a obter as dimensões corretas em cada camada.

In [ ]:
class Generator(nn.Module):
    """
    Defines the generator network of the DCGAN technique.
    """
    def __init__(self):
        """
        Constructor of the generator.
        """
        super(Generator, self).__init__()
        raise NotImplementedError() # Delete this line
        self.network = nn.Sequential(
            # in: latent_size x 1 x 1
            nn.ConvTranspose2d(latent_size, 8 * size_multiplier, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(8 * size_multiplier),
            nn.ReLU(True),
            # tensor: 512 x 4 x 4
            nn.ConvTranspose2d(8 * size_multiplier, 4 * size_multiplier, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(4 * size_multiplier),
            nn.ReLU(True),
            # tensor: 256 x 8 x 8
            # Implement the next layers:
            # tensor: 128 x 16 x 16
            # tensor: 64 x 32 x 32
            # WRITE YOUR CODE HERE! (you can delete this comment)
            nn.ConvTranspose2d(size_multiplier, num_color_channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh(),
            # out: 3 x 64 x 64
        )

    def forward(self, input : torch.Tensor) -> torch.Tensor:
        """
        Defines the forward pass of the generator.
        Args:
            input: the input tensor of shape (batch_size, latent_size, 1, 1)
        Returns:
            The output tensor of shape (batch_size, num_channels, image_size, image_size)
        """
        return self.network(input)

In [ ]:
# Prints the generator network
generator = Generator().to(device)
print(generator)

In [ ]:
def count_parameters(model):
    """
    Auxiliary function to count the number of trainable parameters of a PyTorch model.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
device = torch.device('cpu')

reset_seeds()

generator = Generator().to(device)

assert count_parameters(generator) == 3576704

latent = torch.randn(batch_size, latent_size, 1, 1, device=device)

output = generator.forward(latent)

assert output.shape[0] == batch_size
assert output.shape[1] == num_color_channels
assert output.shape[2] == image_size
assert output.shape[3] == image_size

## Implemente o Discriminador

Na célula a seguir, implemente o discriminador.

Instruções:
- A arquitetura da rede é baseada na arquitetura DCGAN, mas não é exatamente a mesma.
- Cinco camadas convolucionais são utilizadas para transformar a imagem de entrada em um escalar que codifica a probabilidade da imagem ser real.
- Cada camada convolucional utiliza tamanho de kernel igual a 4.
- O stride da última camada convolucional é 1, enquanto as demais utilizam stride igual a 2.
- Nenhum padding é utilizado na última camada convolucional, mas as anteriores utilizam padding igual a 1.
- Nenhum bias é utilizado nas camadas convolucionais.
- As camadas intermediárias utilizam a função de ativação LeakyReLU com inclinação negativa de 0.2, enquanto a última utiliza Sigmoid.
- Batch Normalization é utilizada após cada camada convolucional, exceto a última.
- Utilize `size_multiplier` para ajudá-lo a obter as dimensões corretas em cada camada.

In [ ]:
class Discriminator(nn.Module):
    """
    Defines the discriminator network of the DCGAN technique.
    """
    def __init__(self):
        """
        Constructor of the discriminator.
        """
        super(Discriminator, self).__init__()
        raise NotImplementedError() # Delete this line
        self.network = nn.Sequential(
            # in: 3 x 64 x 64
            nn.Conv2d(num_color_channels, size_multiplier, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(size_multiplier),
            nn.LeakyReLU(0.2, inplace=True),
            # tensor: 64 x 32 x 32
            # Implement the next layers:
            # tensor: 128 x 16 x 16
            # tensor: 256 x 32 x 32
            # tensor: 512 x 4 x 4
            # WRITE YOUR CODE HERE! (you can delete this comment)
            nn.Conv2d(8 * size_multiplier, 1, kernel_size=4, stride=1, padding=0, bias=False),
            # out: 1 x 1 x 1
            nn.Flatten(),
            nn.Sigmoid(),
        )

    def forward(self, input : torch.Tensor) -> torch.Tensor:
        """
        Defines the forward pass of the discriminator.
        Args:
            input: the input tensor of shape (batch_size, num_channels, image_size, image_size)
        Returns:
            The output tensor of shape (batch_size, 1)
        """
        return self.network(input)

In [ ]:
# Prints the discriminator network

discriminator = Discriminator().to(device)
print(discriminator)

In [ ]:
device = torch.device('cpu')

reset_seeds()

discriminator = Discriminator().to(device)

assert count_parameters(discriminator) == 2765696

random_img = torch.randn(batch_size, num_color_channels, image_size, image_size, device=device)

output = discriminator.forward(random_img)

assert output.shape[0] == batch_size
assert output.shape[1] == 1


## Inicializando os pesos

Na DCGAN, os pesos são inicializados de uma maneira particular. Os pesos das camadas convolucionais e de convolução transposta são inicializados com uma distribuição Normal de média zero e desvio padrão igual a 0.02. Note que, neste caso, as camadas convolucionais e de convolução transposta não possuem bias.

In [ ]:
def weights_init(m):
    """
    Initializes the weights of a given layer following the DCGAN convention.
    param m: the layer to be initialized.
    """
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        # WRITE YOUR CODE HERE! (you can delete this comment, but do not delete this cell so the ID is not lost)
        raise NotImplementedError()
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
device = torch.device('cpu')

reset_seeds()

discriminator = Discriminator().to(device)
generator = Generator().to(device)

discriminator.apply(weights_init)
generator.apply(weights_init)

assert abs(torch.sum(discriminator.network[0].weight.data) - (-1.0316)) < 1e-3
assert abs(torch.sum(discriminator.network[1].weight.data) - 64.0033) < 1e-3
assert abs(torch.sum(generator.network[0].weight.data) - 5.0504) < 1e-3
assert abs(torch.sum(generator.network[1].weight.data) - 512.3093) < 1e-3

In [ ]:
sample_dir = 'generated'
os.makedirs(sample_dir, exist_ok=True)

def save_samples(index : int, generator : nn.Module, latent_tensors : torch.Tensor, show=True):
    """
    Auxiliary function to save the generated samples during training.
    Args:
        index: the index of the current epoch, used to name the generated image.
        generator: the generator model.
        latent_tensors: a batch of latent tensors to be fed to the generator to produce the generated images.
        show: whether to show the generated images or not.
    """
    fake_images = generator(latent_tensors)
    fake_fname = 'generated_images_{0:0=4d}.png'.format(index)
    save_image(fake_images, os.path.join(sample_dir, fake_fname), nrow=8, padding=2, normalize=True)
    print('Saving ', fake_fname)
    if show:
        fig, ax = plt.subplots(figsize=(8,8))
        ax.set_xticks([])
        ax.set_yticks([])
        image_grid = make_grid(fake_images.cpu().detach(), nrow=8, normalize=True, value_range=(-1, 1))
        ax.imshow(image_grid.permute(1, 2, 0))
        plt.axis('off')
        plt.show()

## Implementando o Treinamento do Discriminador

Para treinar o discriminador, utilizamos a seguinte função de perda:
\begin{equation}
J^{(D)} = -\mathbb{E}_{x \sim p_{\mathrm{data}}} \log D(x) - \mathbb{E}_z \log \left( 1 - D(G(z)) \right),
\end{equation}
em que o primeiro e o segundo termos são as perdas relacionadas às imagens reais e falsas (isto é, geradas pelo gerador), respectivamente. Utilizando essa função de perda, implemente uma iteração do treinamento do discriminador na célula a seguir.

Dicas:
- Utilize `F.binary_cross_entropy(predictions, targets)` para calcular a entropia cruzada binária entre `predictions` e `targets`.
- Para aprender como gerar imagens falsas, observe o teste da rede geradora.
- Utilize o código relacionado ao cálculo da perda das imagens reais como modelo.

In [ ]:
def training_step_discriminator(discriminator : nn.Module,
                                generator : nn.Module,
                                real_images : torch.Tensor,
                                opt_d : torch.optim.Optimizer) -> tuple[float, float, float]:
    """
    Executes an iteration of the discriminator's training.
    param discriminator: the discriminator's model.
    param generator: the generator's model.
    param real_images: real images from the dataset.
    param opt_d: the optimizer used to execute a step of Gradient Descent.
    return: three value are returned: the total loss, the score from the
            real images, and the score from the fake images.
    """
    opt_d.zero_grad()

    real_preds = discriminator(real_images)
    real_targets = torch.ones(real_images.size(0), 1, device=device)
    real_loss = F.binary_cross_entropy(real_preds, real_targets)
    real_score = torch.mean(real_preds).item()

    # Create fake images and their targets
    # Compute the fake loss using binary cross-entropy
    # To learn how to generate fake images, see how the generator was tested above
    # Use how the real loss is computed as a template
    # WRITE YOUR CODE HERE! (you can delete this comment, but do not delete this cell so the ID is not lost)
    raise NotImplementedError() # Delete this line

    loss = real_loss + fake_loss
    loss.backward()
    opt_d.step()

    return loss.item(), real_score, fake_score

In [ ]:
device = torch.device('cpu')

reset_seeds()

generator = Generator().to(device)
discriminator = Discriminator().to(device)

discriminator.apply(weights_init)
generator.apply(weights_init)

opt_d = torch.optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(beta1, beta2))

latent = torch.randn(batch_size, latent_size, 1, 1, device=device)

real_images = generator(latent) # Using them as real images, but they are fake
loss_d, real_score, fake_score = training_step_discriminator(discriminator, generator, real_images.to(device), opt_d)

assert abs(loss_d - 1.8112266063690186) < 1e-3
assert abs(real_score - 0.6484211087226868) < 1e-3
assert abs(fake_score - 0.6575933694839478) < 1e-3

## Implementando o Treinamento do Gerador

Para treinar o gerador, utilize como função de perda:
\begin{equation}
J^{(G)} = -\mathbb{E}_z \log \left( D(G(z)) \right).
\end{equation}

Utilizando essa função de perda, implemente uma iteração do treinamento do gerador na célula a seguir.

Dicas:
- Utilize `F.binary_cross_entropy(predictions, targets)` para calcular a entropia cruzada binária entre `predictions` e `targets`.
- Para aprender como gerar imagens falsas, observe o teste da rede geradora.
- Utilize o código relacionado ao cálculo da perda das imagens reais na etapa de treinamento do discriminador como modelo.

In [ ]:
def training_step_generator(discriminator : nn.Module, generator : nn.Module, num_images : int, opt_g : torch.optim.Optimizer) -> float:
    """
    Executes an iteration of the discriminator's training.
    param discriminator: the discriminator's model.
    param generator: the generator's model.
    param num_images: the number of images to generate.
    param opt_g: the optimizer used to execute a step of Gradient Descent.
    return: the loss value.
    """
    opt_g.zero_grad()

    # WRITE YOUR CODE HERE! (you can delete this comment, but do not delete this cell so the ID is not lost)
    raise NotImplementedError()

    loss.backward()
    opt_g.step()

    return loss.item()

In [ ]:
device = torch.device('cpu')

reset_seeds()

generator = Generator().to(device)
discriminator = Discriminator().to(device)

discriminator.apply(weights_init)
generator.apply(weights_init)

opt_g = torch.optim.Adam(generator.parameters(), lr=learning_rate, betas=(beta1, beta2))

latent = torch.randn(batch_size, latent_size, 1, 1, device=device)

loss_g = training_step_generator(discriminator, generator, batch_size, opt_g)

assert abs(loss_g - 0.49504929780960083) < 1e-3

In [ ]:
# Generating a fixed latent vector so we can compare results from different epochs
reset_seeds()

fixed_latent = torch.randn(batch_size, latent_size, 1, 1, device=device)

### Implementando o Treinamento

Como mencionado em aula, o treinamento do discriminador e do gerador é intercalado. Na função a seguir, implemente a etapa de treinamento considerando que primeiro executamos uma etapa de treinamento do discriminador e, em seguida, uma etapa de treinamento do gerador. **Dica:** observe os testes das funções de etapa de treinamento.

In [ ]:
def fit_step(discriminator : nn.Module, generator : nn.Module,
             real_images : torch.Tensor,
             opt_d : torch.optim.Optimizer,
             opt_g : torch.optim.Optimizer) -> tuple[float, float, float, float]:
    """
    Executes a step of the DCGAN training.
    :param discriminator: the discriminator's model.
    :param generator: the generator's model.
    :param real_images: the real images.
    :param opt_d: the discriminator's optimizer.
    :param opt_g: the generator's optimizer.
    :return: the discriminator's loss, the generator's loss, the score related to the real images, and
             the score related to the fake images.
    """
    # WRITE YOUR CODE HERE! (you can delete this comment, but do not delete this cell so the ID is not lost)
    raise NotImplementedError()
    return loss_d, loss_g, real_score, fake_score

In [ ]:
device = torch.device('cpu')

reset_seeds()

generator = Generator().to(device)
discriminator = Discriminator().to(device)

discriminator.apply(weights_init)
generator.apply(weights_init)

real_images, _ = next(iter(train_dl))
fit_step(discriminator, generator, real_images, opt_d, opt_g)

assert abs(torch.sum(discriminator.network[0].weight.data) - (-1.0316)) < 1e-3
assert abs(torch.sum(discriminator.network[1].weight.data) - 64.0033) < 1e-3
assert abs(torch.sum(generator.network[0].weight.data) - 5.0504) < 1e-3
assert abs(torch.sum(generator.network[1].weight.data) - 512.3093) < 1e-3

In [ ]:
def fit(num_epochs : int, learning_rate : float, show : bool = False):
    """
    Trains the DCGAN.
    param num_epochs: number of epochs used in the training.
    param learning_rate: learning rate used for the optimizers.
    param show: if the training results should be shown during training.
    """
    torch.cuda.empty_cache()

    # Create the optimizers
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(beta1, beta2))
    opt_g = torch.optim.Adam(generator.parameters(), lr=learning_rate, betas=(beta1, beta2))

    history = np.zeros((num_epochs, 4)) # to store the history of losses and scores during training

    for epoch in range(num_epochs):
        for real_images, _ in tqdm(train_dl):
            # Executes a step of the training
            loss_d, loss_g, real_score, fake_score = fit_step(discriminator, generator, real_images.to(device), opt_d, opt_g)
            history[epoch, :] = loss_d, loss_g, real_score, fake_score

        # Log losses and scores from the last batch
        print('Epoch [{}/{}], loss_g: {:.4f}, loss_d: {:.4f}, real_score: {:.4f}, fake_score: {:.4f}'.format(
            epoch + 1, num_epochs, loss_g, loss_d, real_score, fake_score
        ))

        save_samples(epoch + 1, generator, fixed_latent, show)

    return history

O código a seguir treina a rede por 1 época. Ao final da época, gera-se um mosaico de imagens de gatos usando o gerador.

A imagem gerada é salva numa pasta chamada `generated` em seu computador ou no Google Colab. **Inclua a imagem gerada no seu relatório.**

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

reset_seeds()

fixed_latent = torch.randn(batch_size, latent_size, 1, 1, device=device)

generator = Generator().to(device)
discriminator = Discriminator().to(device)

discriminator.apply(weights_init)
generator.apply(weights_init)

train_dl = DataLoader(train_ds, batch_size, shuffle=True, num_workers=2)

_ = fit(num_epochs=1, learning_rate=learning_rate, show=True)

## Realizando o Treinamento

Treinar por apenas uma época produz resultados que se parecem com imagens tênues de gatos. Para obter melhor qualidade, precisamos treinar por mais tempo. A célula a seguir treina a GAN por 50 épocas.

Você definitivamente precisará de uma GPU para executar este treinamento. Esse treinamento é opcional, mas recomendo fortemente que você o execute para ver os resultados. É possível obter uma qualidade muito melhor com ainda mais treinamento.

Para visualizar os resultados, procure por uma pasta chamada `generated` em seu computador ou no Google Colab. Ao final de cada iteração, gera-se um mosaico de imagens de gatos.

Mesmo com 50 épocas, os gatos ainda parecerão pequenos demônios peludinhos fofinhos 😹. Você também poderá observar a evolução ao longo das épocas.

Finalmente, traça-se gráficos das perdas e dos *scores* ao longo das épocas.

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

reset_seeds()

fixed_latent = torch.randn(batch_size, latent_size, 1, 1, device=device)

generator = Generator().to(device)
discriminator = Discriminator().to(device)

discriminator.apply(weights_init)
generator.apply(weights_init)

train_dl = DataLoader(train_ds, batch_size, shuffle=True, num_workers=2)

num_epochs = 50
history = fit(num_epochs=num_epochs, learning_rate=learning_rate, show=False)

plt.figure()
plt.plot(history[:, 0], label='Discriminator Loss')
plt.plot(history[:, 1], label='Generator Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Losses during training')
plt.legend()
plt.savefig('losses.png')
plt.show()

plt.figure()
plt.plot(history[:, 2], label='Real Score')
plt.plot(history[:, 3], label='Fake Score')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('Scores during training')
plt.legend()
plt.savefig('scores.png')
plt.show()

Os resultados do treinamento (opcional) da célula acima são mostrados a seguir.

![generated_images_0050.png](attachment:generated_images_0050.png)

**Figura 1:** Imagens de gatos (?) geradas pelo gerador após um treinamento de 50 épocas.

![losses.png](attachment:losses.png)

**Figura 2:** Perdas do discriminador e do gerador ao longo de um treinamento de 50 épocas.

![scores.png](attachment:scores.png)

**Figura 3:** *Scores* do discriminador ao longo de um treinamento de 50 épocas.

# 3. Entrega

A entrega consiste do notebook no formato **.ipynb** e de um relatório, submetida através do Google Classroom. Modificações nos arquivos do código base são permitidas, desde que o nome e a interface dos scripts “main” não sejam alterados. A princípio, não há limitação de número de páginas para o relatório, mas pede-se que seja sucinto. O relatório deve conter:
- Figuras que comprovem o funcionamento do seu código.
- Demais solicitações feitas ao longo do roteiro.

Por limitações do Google Classroom (e por motivo de facilitar a automatização da correção), entregue seu laboratório com todos os arquivos num único arquivo **.zip** (**não** utilize outras tecnologias de compactação de arquivos) com o seguinte padrão de nome: **“<login_email_google_education>_labX.zip”**. Por exemplo, no meu caso, meu login Google Education é **marcos.maximo**, logo eu entregaria o lab 1 como **“marcos.maximo_lab1.zip”**. **Não** crie subpastas para os arquivos da sua entrega, **deixe todos os arquivos na “raiz” do .zip**.